# Dense-navigation semantic HNSW: gate + bridge experiment

This is the next experiment after validating parity of our custom HNSW walk.

The key change is: **semantic score no longer steers navigation priority**. HNSW navigation stays dense-only (`lambda = 0`). The semantic program only decides whether a node is eligible as a result; failed nodes remain traversable and can expose bounded 2-hop bridge candidates.

We compare, at controlled semantic selectivities:

1. `hnsw_postfilter`
2. `hnsw_filtered` from `hnsw_rs`
3. `custom_hnsw_lambda0` — our dense filtered parity baseline, no bridge hops
4. `dense_semantic_bridge` — our dense navigation + semantic gate + bounded 2-hop bridge expansion

We sweep the fraction of the catalog that passes the semantic query. This is important because the previous ~48% pass rate was an easy filtering regime.

**Timing scope:** semantic scores are still precomputed before each Rust query loop. This notebook isolates traversal/filter/bridge behavior. It counts semantic evaluations so the next experiment can fuse the real 216-byte bitwise program live.


In [ ]:
#@title 1) Settings
FULL_DATA = False #@param {type:"boolean"}
QUERIES = 100 #@param {type:"integer"}
K = 50 #@param {type:"integer"}
EF = 128 #@param {type:"integer"}
M = 24 #@param {type:"integer"}
EF_CONSTRUCTION = 200 #@param {type:"integer"}
BRIDGE_HOPS = 2 #@param {type:"integer"}
BRIDGE_CAP = 64 #@param {type:"integer"}
POSTFILTER_OVERSAMPLE = 8 #@param {type:"integer"}
POSITIVE = 'minimalist,office_appropriate' #@param {type:"string"}
NEGATIVE = 'technical_sporty' #@param {type:"string"}
TARGET_FRACTIONS = '0.50,0.20,0.10,0.05,0.02,0.01' #@param {type:"string"}

print({
    'FULL_DATA': FULL_DATA, 'QUERIES': QUERIES, 'K': K, 'EF': EF,
    'BRIDGE_HOPS': BRIDGE_HOPS, 'BRIDGE_CAP': BRIDGE_CAP,
    'POSITIVE': POSITIVE, 'NEGATIVE': NEGATIVE,
    'TARGET_FRACTIONS': TARGET_FRACTIONS,
})


In [ ]:
#@title 2) Clone repo + install Python/Rust dependencies
import os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/ras')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','faiss-cpu'], check=True)

if shutil.which('rustc') is None or shutil.which('cargo') is None:
    print('Installing minimal stable Rust toolchain...')
    subprocess.run(['bash','-lc', "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], check=True)
    cargo_bin = str(pathlib.Path.home()/'.cargo'/'bin')
    os.environ['PATH'] = cargo_bin + os.pathsep + os.environ.get('PATH','')

print('commit:', subprocess.check_output(['git','rev-parse','HEAD']).decode().strip())
print('python:', sys.version.split()[0])
print('rustc:', subprocess.check_output(['rustc','--version']).decode().strip())


In [ ]:
#@title 3) Export real fashion embeddings + compiled semantic programs
import pathlib, shutil, subprocess, sys, time, os
os.chdir('/content/ras')
CFG = 'configs/binary_bbq.yaml' if FULL_DATA else 'configs/binary_bbq_smoke.yaml'
ASSETS = pathlib.Path('/content/semantic_hnsw_dense_bridge_assets')
if ASSETS.exists():
    shutil.rmtree(ASSETS)
t0 = time.time()
subprocess.run([sys.executable,'-m','experiments.export_native_finalists','--config',CFG,'--out-dir',str(ASSETS)], check=True)
print(f'export finished in {(time.time()-t0)/60:.1f} min')
print('assets:', ASSETS)
print('programs:', sorted(p.name for p in (ASSETS/'sidecar_programs').iterdir() if p.is_dir()))


In [ ]:
#@title 4) Compile semantic-HNSW Rust executable
import subprocess, os, time
os.chdir('/content/ras')
t0 = time.time()
subprocess.run(['cargo','build','--release','--manifest-path','rust/semantic_engine/Cargo.toml','--bin','semantic_hnsw'], check=True)
BIN = '/content/ras/rust/semantic_engine/target/release/semantic_hnsw'
print(f'compiled in {time.time()-t0:.1f}s')
print(BIN)


In [ ]:
#@title 5) Convert target selectivities into semantic gates
import numpy as np, pandas as pd, sys, pathlib, importlib

# Force the checked-out package in case Colab has another `ras` namespace.
SRC = str(pathlib.Path('/content/ras/src'))
sys.path[:] = [p for p in sys.path if p != SRC]
sys.path.insert(0, SRC)
for name in list(sys.modules):
    if name == 'ras' or name.startswith('ras.'):
        del sys.modules[name]
importlib.invalidate_caches()
from ras import SemanticExecutor

pos = [x.strip() for x in POSITIVE.split(',') if x.strip()]
neg = [x.strip() for x in NEGATIVE.split(',') if x.strip()]
n_pred = len(pos) + len(neg)
executor = SemanticExecutor.open(str(ASSETS/'sidecar_index'), str(ASSETS/'sidecar_programs'))
n_items = executor.index.n_items
ids = np.arange(n_items, dtype=np.int64)
# Rust uses mean log-probability across requested predicates.
sem_mean = executor.score_candidates(ids, positive=pos, negative=neg) / max(1, n_pred)

requested = [float(x.strip()) for x in TARGET_FRACTIONS.split(',') if x.strip()]
usable = []
for f in requested:
    if not (0 < f <= 1):
        continue
    if f * n_items < K + 1:
        print(f'skipping target fraction {f:.3f}: only ~{f*n_items:.1f} eligible items for K={K}')
        continue
    gate = float(np.quantile(sem_mean, 1.0 - f))
    actual = float(np.mean(sem_mean >= gate))
    usable.append({'target_fraction': f, 'gate_logprob': gate, 'actual_fraction_python': actual, 'eligible_items': int((sem_mean >= gate).sum())})
gate_df = pd.DataFrame(usable).sort_values('target_fraction', ascending=False).reset_index(drop=True)
display(gate_df)
assert len(gate_df), 'No target fraction leaves enough eligible items for K; reduce K or use more data.'


In [ ]:
#@title 6) Run dense-navigation bridge search across selectivities
import subprocess, pathlib, time, pandas as pd

all_runs = []
for r in gate_df.itertuples(index=False):
    frac = float(r.target_fraction)
    gate = float(r.gate_logprob)
    out = pathlib.Path(f'/content/semantic_hnsw_dense_bridge_{frac:.3f}.csv')
    cmd = [
        BIN,
        '--assets', str(ASSETS),
        '--programs', str(ASSETS/'sidecar_programs'),
        '--positive', POSITIVE,
        '--negative', NEGATIVE,
        '--queries', str(QUERIES),
        '--k', str(K),
        '--ef', str(EF),
        '--m', str(M),
        '--ef-construction', str(EF_CONSTRUCTION),
        # Dense-only navigation: semantics do NOT change frontier priority.
        '--semantic-lambda', '0.0',
        '--gate-logprob', str(gate),
        '--bridge-hops', str(BRIDGE_HOPS),
        '--bridge-cap', str(BRIDGE_CAP),
        '--postfilter-oversample', str(POSTFILTER_OVERSAMPLE),
        '--out', str(out),
    ]
    print(f'\n=== target qualified fraction {frac:.3f}, gate {gate:.5f} ===')
    t0 = time.time()
    run = subprocess.run(cmd, text=True, capture_output=True)
    print(run.stdout)
    if run.returncode != 0:
        print(run.stderr)
        raise RuntimeError(f'semantic_hnsw failed for fraction {frac} with code {run.returncode}')
    z = pd.read_csv(out)
    z['target_fraction'] = frac
    z['gate_logprob'] = gate
    # In this run semantic_hnsw_c has lambda=0, so give it the actual algorithmic name.
    z['method'] = z['method'].replace({'semantic_hnsw_c': 'dense_semantic_bridge'})
    all_runs.append(z)
    print(f'wall time: {time.time()-t0:.2f}s')

results = pd.concat(all_runs, ignore_index=True)
print('rows:', len(results))


In [ ]:
#@title 7) Recall / latency / traversal summary
import numpy as np, pandas as pd
summary = (results.groupby(['target_fraction','method'])
    .agg(
        queries=('query_id','count'),
        mean_latency_ms=('latency_ms','mean'),
        p50_latency_ms=('latency_ms','median'),
        p95_latency_ms=('latency_ms', lambda x: np.quantile(x, .95)),
        mean_recall_at_k=('recall_at_k','mean'),
        mean_returned=('returned','mean'),
        mean_visited=('visited','mean'),
        mean_semantic_evals=('semantic_evals','mean'),
        mean_bridge_candidates=('bridge_candidates','mean'),
        qualified_fraction=('qualified_fraction','mean'),
    )
    .reset_index())
display(summary.sort_values(['target_fraction','mean_latency_ms'], ascending=[False,True]))

# Directly compare our dense+bridge method to filtered HNSW at each selectivity.
piv_r = summary.pivot(index='target_fraction', columns='method', values='mean_recall_at_k')
piv_t = summary.pivot(index='target_fraction', columns='method', values='mean_latency_ms')
rows = []
for f in piv_r.index:
    row = {'target_fraction': f}
    if 'dense_semantic_bridge' in piv_r.columns and 'hnsw_filtered' in piv_r.columns:
        row['bridge_minus_filtered_recall'] = float(piv_r.loc[f,'dense_semantic_bridge'] - piv_r.loc[f,'hnsw_filtered'])
        row['bridge_over_filtered_latency'] = float(piv_t.loc[f,'dense_semantic_bridge'] / piv_t.loc[f,'hnsw_filtered'])
    if 'custom_hnsw_lambda0' in piv_r.columns:
        row['bridge_minus_custom_no_bridge_recall'] = float(piv_r.loc[f,'dense_semantic_bridge'] - piv_r.loc[f,'custom_hnsw_lambda0'])
    rows.append(row)
comparison = pd.DataFrame(rows).sort_values('target_fraction', ascending=False)
display(comparison)
print('NOTE: timing still excludes live bitwise semantic-program execution; semantic_evals tells us how much work must be fused next.')


In [ ]:
#@title 8) Plot the selectivity-dependent frontier
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,6))
for method, g in summary.groupby('method'):
    g = g.sort_values('target_fraction', ascending=False)
    ax.plot(g['mean_latency_ms'], g['mean_recall_at_k'], marker='o', label=method)
    for _, r in g.iterrows():
        ax.annotate(f"{100*r['target_fraction']:.0f}%", (r['mean_latency_ms'], r['mean_recall_at_k']), xytext=(4,4), textcoords='offset points', fontsize=8)
ax.set_xlabel('Mean query latency (ms)')
ax.set_ylabel(f'Mean Recall@{K}')
ax.set_title('Semantic HNSW: recall-latency frontier vs predicate selectivity')
ax.legend()
ax.grid(True, alpha=.25)
plt.show()


In [ ]:
#@title 9) Package results
import pathlib, shutil, json, platform, subprocess
PKG = pathlib.Path('/content/semantic_hnsw_dense_bridge_artifact')
if PKG.exists(): shutil.rmtree(PKG)
PKG.mkdir()
results.to_csv(PKG/'per_query.csv', index=False)
summary.to_csv(PKG/'summary.csv', index=False)
comparison.to_csv(PKG/'comparison.csv', index=False)
gate_df.to_csv(PKG/'gates.csv', index=False)
meta = {
  'commit': subprocess.check_output(['git','-C','/content/ras','rev-parse','HEAD']).decode().strip(),
  'full_data': FULL_DATA, 'queries': QUERIES, 'k': K, 'ef': EF, 'm': M,
  'ef_construction': EF_CONSTRUCTION, 'semantic_lambda': 0.0,
  'bridge_hops': BRIDGE_HOPS, 'bridge_cap': BRIDGE_CAP,
  'postfilter_oversample': POSTFILTER_OVERSAMPLE, 'positive': POSITIVE, 'negative': NEGATIVE,
  'target_fractions': gate_df.target_fraction.tolist(),
  'python': platform.python_version(), 'platform': platform.platform(),
  'cpu': pathlib.Path('/proc/cpuinfo').read_text().split('model name')[1].split('\n')[0].split(':',1)[-1].strip() if pathlib.Path('/proc/cpuinfo').exists() and 'model name' in pathlib.Path('/proc/cpuinfo').read_text() else 'unknown',
  'timing_scope': 'Dense HNSW traversal + precomputed semantic gate + bridge behavior; live 216-byte predicate execution is not yet timed',
}
(PKG/'environment.json').write_text(json.dumps(meta, indent=2))
shutil.make_archive('/content/rsa_semantic_hnsw_dense_bridge','zip',PKG)
print('/content/rsa_semantic_hnsw_dense_bridge.zip')
